This notebook will be used to test reading pdfs directly with LLMs

In [1]:
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv
from llm_maker import make_llm

load_dotenv()
llm = make_llm(model="pixtral-large", temperature=0.0)


In [2]:
from typing import Generic, TypeVar, Optional, Literal
from pydantic import BaseModel, Field

T = TypeVar("T")

class Confident(BaseModel, Generic[T]):
    """A field value paired with the model's self-reported confidence and evidence.

    - value:      the extracted value (typed T)
    - confidence: 0..1 self-reported certainty this value is correct given the source
    - evidence:   verbatim snippet from the source text supporting the value
                  (used to verify the value is grounded, not hallucinated)
    """
    value: T
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="Your certainty (0-1) that `value` is correct given ONLY the source text.",
    )
    evidence: str = Field(
        description="Short verbatim snippet copied from the source text that supports `value`.",
    )


In [3]:
# ---------------------------------------------------------------------------
# Node 1 — Document Segmentation
#
# These models are used ONLY to identify the boundaries of the independent
# accounting documents contained in a single PDF. They do NOT extract any
# invoice information. Extraction (ClassifyExtract) happens later, once per
# split document.
#
# IMPORTANT: the total page count is NOT part of this schema. A VLM cannot
# reliably count PDF pages, so the count is computed deterministically with
# pypdf and passed INTO the prompt. The model only assigns boundaries within
# that known 1..total_pages range.
# ---------------------------------------------------------------------------


class DocumentBoundary(BaseModel):
    """
    Boundary of a single independent accounting document inside a PDF.

    Pages are 1-based and inclusive on both ends.
    """

    start_page: int = Field(
        description="1-based page where this document starts (inclusive)."
    )
    end_page: int = Field(
        description="1-based page where this document ends (inclusive)."
    )
    confidence: float = Field(
        ge=0.0,
        le=1.0,
        description="Certainty (0-1) that these boundaries are correct.",
    )


class DocumentSegmentation(BaseModel):
    """
    Segmentation result: the independent accounting documents found in the
    PDF, in reading order.

    The total page count is known deterministically and is NOT part of this
    schema — the model only returns boundaries.
    """

    documents: list[DocumentBoundary] = Field(
        description="Independent accounting documents contained in the PDF, in page order."
    )


In [4]:


class DocumentClassification(BaseModel):
    """
    Semantic classification of the document.

    The goal is to identify the accounting nature of the document, independently
    from its visual appearance.
    """

    document_type: Confident[
        Literal[
            "invoice",
            "receipt",
            "credit_note",
            "debit_note",
            "other"
        ]
    ] = Field(
        description="""
    Primary accounting document type.

    Choose:

    - invoice
    - receipt
    - credit_note
    - debit_note
    - other

    Use 'other' whenever the document is not one of the above, even if it contains
    financial information (shipping documents, customs documents, bank statements,
    insurance certificates, purchase orders, etc.).

    Do not infer the type from filenames.
    Only use the document contents.
    """
    )

    document_state: Optional[
        Confident[
            Literal[
                "original",
                "proforma",
                "copy",
                "cancelled"
            ]
        ]
    ] = Field(
        default=None,
        description="""
    Legal status of the document.

    Examples:

    Proforma Invoice (provisional invoice)
    Original
    Cópia
    Duplicate (copy)
    Duplicado (copy)
    ANULADO (cancelled)
    Cancelled (cancelled)

    Return null when no state is explicitly indicated.
    """
    )

In [5]:
class SemanticBlock(BaseModel):
    """
    A semantically meaningful block of text extracted from the document.

    A block should correspond to one logical section of the document
    (for example the invoice totals, VAT summary or line-item table),
    rather than an arbitrary OCR chunk.

    These blocks are intended for downstream deterministic parsing and
    validation. They should preserve the textual structure of the original
    document as much as possible.
    """

    kind: Literal[
        "header",
        "line_items",
        "vat_summary",
        "totals",
        "footer",
        "other",
    ] = Field(
        description="""
    Semantic type of this text block.

    Choose the best matching category:

    - header:
        Supplier/customer information, invoice number,
        invoice date, document identifiers, payment terms,
        document title, addresses.

    - line_items:
        The commercial table describing products or services.
        Preserve every row exactly as shown whenever possible.

    - vat_summary:
        VAT / tax breakdown tables.
        Includes taxable bases, VAT percentages,
        exemption reasons and VAT totals.

    - totals:
        Final monetary summary.
        Includes subtotal, discounts, VAT total,
        grand total, amount payable,
        currency and similar summary fields.

    - footer:
        Legal notes, payment instructions,
        bank information, disclaimers,
        QR references, certification statements.

    - other:
        Any meaningful block that does not fit the
        previous categories.
    """
    )

    page: int = Field(
        description="""
    1-based page number on which this block appears.
    """
        )

    text: str = Field(
        description="""
    Text contained in this semantic block.

    Guidelines:

    • Preserve line breaks whenever they help retain table structure.

    • Preserve row ordering.

    • Preserve negative signs.

    • Preserve decimal values exactly.

    • Do NOT rewrite, summarize or normalize.

    • Exclude unrelated neighbouring sections.

    Examples of a line_items block:

    Gasóleo Simples
    54,99   1,5878   81,95   23%   18,85
    56,04   1,5187   79,65   23%   18,32

    or

    REMUNERATION EXPORT CARGO
    2     52,00     104,00

    Examples of a totals block:

    Subtotal      507,22
    VAT            55,99
    Total         563,21
"""
    )

In [6]:
class InvoiceData(BaseModel):
    """
    Core accounting information extracted from an invoice-like document.

    Every field is optional.

    Never guess.
    Prefer null whenever multiple values could match.
    """

    supplier_name: Optional[Confident[str]] = Field(
        default=None,
        description="""
        Legal or commercial name of the company issuing the invoice.

        Usually located:

        - top header
        - company logo
        - company address block

        This is the seller.

        Do NOT return:

        - customer
        - recipient
        - ship-to
        - consignee
        - billing contact
        """
    )

    supplier_vat: Optional[Confident[str]] = Field(
        default=None,
        description="""
        Tax identification number (VAT/NIF) of the supplier.

        Usually appears close to the supplier name.

        Examples:

        PT501925350
        PT500697370
        NL813715532B01

        For Portuguese invoices:

        If exactly nine digits are shown without country prefix,
        normalize as:

        PT#########

        Do not invent prefixes for other countries.
        """
    )


    client_name: Optional[Confident[str]] = Field(
        default=None,
        description="""
        Company or person receiving the invoice.

        Usually located in:

        - customer block
        - invoice address
        - 'Exmo(s). Senhor(es)'
        - Bill To
        - Customer

        Do not confuse with supplier.
        """
    )

    client_vat: Optional[Confident[str]] = Field(
        default=None,
        description="""
        VAT number of the customer.

        Usually appears close to the customer name.

        Normalize Portuguese NIFs to PT######### when only nine digits
        are shown.

        Return null if no customer VAT appears.
        """
    )

    issue_date: Optional[Confident[str]] = Field(
        default=None,
        description="""
        Invoice issue date.

        Normalize to:

        DD-MM-YYYY

        Look for labels such as:

        Issue Date
        Data
        Data de emissão
        Invoice Date

        Do NOT use:

        Due Date
        Shipment Date
        Check-in
        Check-out
        Operation Date
        Service Period
        """
    )

    base_amount: Optional[Confident[float]] = Field(
        default=None,
        description="""
        Net amount before VAT.

        Usually labelled:

        Base
        Valor Líquido
        Net Amount
        Taxable Amount
        Subtotal

        Do not return the total including VAT.

        When VAT is zero due to exemption or reverse charge,
        this value may equal the total amount.
        """
    )

    vat_amount: Optional[Confident[float]] = Field(
        default=None,
        description="""
        Total VAT amount charged on the invoice.

        This is an absolute monetary amount.

        Examples:

        55.99
        189.60
        0.00

        Do NOT return:

        23
        16
        6

        Those are VAT rates, not VAT amounts.
        """
    )

    total_amount: Optional[Confident[float]] = Field(
        default=None,
        description="""
        Final payable amount including VAT.

        Usually labelled:

        Total
        Total Invoice
        Valor Total
        Total da Fatura
        Amount Due

        For zero-VAT invoices:

        total = base amount.

        Do not return intermediate subtotals.
        """
    )

    currency: Optional[Confident[str]] = Field(
        default=None,
        description="""
        Currency code of the invoice.

        Use ISO 4217 three-letter codes:
        EUR, USD, GBP, JPY, etc.
        If no currency is explicitly indicated, return null.
        """
    )

    semantic_blocks: Optional[list[SemanticBlock]] = Field(
        default=None,
        description="""
        Relevant semantic sections extracted from the document.

        These blocks are intended to support downstream validation and
        table parsing without requiring another Vision LLM call.

        Include only blocks useful for accounting validation.

        Typical documents contain between two and five blocks:

        • Header
        • Line items
        • VAT summary
        • Totals

        Do not include repeated pages, advertisements,
        legal disclaimers, company history, marketing content,
        or unrelated attachments unless they contain information
        needed to validate the extracted invoice.

        If a section does not exist (for example an invoice without
        a VAT summary), simply omit that block.
        """
)

In [41]:


T = TypeVar("T")


class ClassifyExtract(BaseModel):

    classification: DocumentClassification = Field(
        description="Semantic classification of the document."
    )

    extracted_data: Optional[InvoiceData] = Field(
        default=None,
        description="""
Populate only when the document type is:

- invoice 
- credit_note

Otherwise return null.
"""
    )

In [44]:
import base64
import io
from pathlib import Path

from pypdf import PdfReader

#Invoices to use in testing
# attachment_path = Path("docs/2150768990.pdf")
# attachment_path = Path("docs/149-2026 CA Indosuez Portugal_Lumiso SGPS Unipessoal LDA_Custody Commission excluding tax Q2 2026.pdf")  
# attachment_path = Path("docs/20260625185316.pdf")  
# attachment_path = Path("docs/FT-4302108.pdf")  
# attachment_path = Path("docs/8940.CCDO.10443.2026.pdf")
attachment_path = Path("docs/Invoice 229555.pdf")

print(attachment_path)

with open(attachment_path, "rb") as f:
    pdf_bytes = f.read()
    encoded_pdf = base64.b64encode(pdf_bytes).decode("utf-8")

# Page count is deterministic — never ask the VLM to count pages.
total_pages = len(PdfReader(io.BytesIO(pdf_bytes)).pages)
print(f"Total pages (deterministic): {total_pages}")


docs\Invoice 229555.pdf
Total pages (deterministic): 1


In [45]:
# ---------------------------------------------------------------------------
# Node 1 — Run Document Segmentation
#
# The VLM cannot count pages reliably, so we DON'T ask it to. We pass the
# deterministic `total_pages` into the prompt and the model only assigns
# boundaries within the known 1..total_pages range.
# ---------------------------------------------------------------------------

segmentation_system_message = SystemMessage(
    content=f"""
You are a document segmentation system for accounting documents.

Your task is ONLY to identify the boundaries of the independent accounting
documents inside the attached PDF.

The PDF has exactly {total_pages} page(s). This number is authoritative — do
NOT re-count the pages and do NOT return page numbers outside the range
1..{total_pages}.

For each document return:

- start_page (1-based, inclusive, between 1 and {total_pages})
- end_page   (1-based, inclusive, between 1 and {total_pages})
- confidence (0-1)

Do NOT extract invoice information (no supplier, amounts, VAT, dates, etc.).

A new document begins whenever a new invoice, receipt, credit note, debit
note or unrelated accounting document starts. 

There can be multiple independent documents in a single PDF, and they may be
of different types.

There can also be multiple copies of the same document. Treat each copy as a
DISTINCT document and return it separately. If there is an original and copy, 
there should be two SEPERATE entries in the output.

Cover every page exactly once: the boundaries must be contiguous and
non-overlapping. The first document must start at page 1 and the last document
must end at page {total_pages}.
"""
)

segmentation_human_message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": (
                f"This PDF has {total_pages} page(s). "
                "Identify the boundaries of the independent accounting documents. "
                f"All page numbers must be between 1 and {total_pages}."
            ),
        },
        {
            "type": "file",
            "file": {
                "file_data": f"data:application/pdf;base64,{encoded_pdf}",
                "filename": attachment_path.name,
                "format": "application/pdf",
            },
        },
    ]
)

segmentation_llm = llm.with_structured_output(DocumentSegmentation)

segmentation = segmentation_llm.invoke([
    segmentation_system_message,
    segmentation_human_message,
])


def validate_segmentation(documents: list[DocumentBoundary], total_pages: int) -> list[str]:
    """Check the VLM boundaries against the deterministic page count.

    Returns a list of human-readable problems (empty == clean coverage).
    """
    problems: list[str] = []
    if not documents:
        return [f"No documents returned for a {total_pages}-page PDF."]

    ordered = sorted(documents, key=lambda d: d.start_page)
    covered: set[int] = set()
    expected_next = 1
    for d in ordered:
        if not (1 <= d.start_page <= total_pages) or not (1 <= d.end_page <= total_pages):
            problems.append(
                f"Out-of-range boundary {d.start_page}-{d.end_page} (valid 1-{total_pages})."
            )
        if d.end_page < d.start_page:
            problems.append(f"Inverted boundary {d.start_page}-{d.end_page}.")
        if d.start_page != expected_next:
            problems.append(
                f"Gap/overlap: expected next document to start at page {expected_next}, "
                f"got {d.start_page}."
            )
        for p in range(max(d.start_page, 1), min(d.end_page, total_pages) + 1):
            if p in covered:
                problems.append(f"Page {p} covered by more than one document.")
            covered.add(p)
        expected_next = d.end_page + 1

    missing = set(range(1, total_pages + 1)) - covered
    if missing:
        problems.append(f"Pages not covered by any document: {sorted(missing)}.")
    return problems


segmentation_problems = validate_segmentation(segmentation.documents, total_pages)

print("Detected documents")
print("------------------")
print(f"Total pages: {total_pages}")
for i, doc in enumerate(segmentation.documents, start=1):
    print(
        f"{i}. pages {doc.start_page}-{doc.end_page}  "
        f"confidence={doc.confidence:.2f}"
    )

if segmentation_problems:
    print("\n⚠️  Segmentation coverage issues:")
    for problem in segmentation_problems:
        print(f"  - {problem}")
else:
    print("\n✅ Boundaries cover all pages exactly once.")


Detected documents
------------------
Total pages: 1
1. pages 1-1  confidence=1.00

✅ Boundaries cover all pages exactly once.


In [46]:
# ---------------------------------------------------------------------------
# Node 2 — Split PDF
#
# If the segmentation found a single document, do nothing (reuse the original
# PDF). Otherwise, split the source PDF into one child PDF per detected
# document, keeping the original page bytes.
#
# Each entry in `split_documents` is a self-contained unit for the extraction
# node: filename + base64 payload + the boundary that produced it.
# ---------------------------------------------------------------------------

from dataclasses import dataclass

from pypdf import PdfWriter


@dataclass
class SplitDocument:
    filename: str
    encoded_pdf: str
    boundary: DocumentBoundary


def split_pdf(pdf_bytes: bytes, base_filename: str, boundaries: list[DocumentBoundary]) -> list[SplitDocument]:
    stem = Path(base_filename).stem
    suffix = Path(base_filename).suffix or ".pdf"

    # Single document -> no split, reuse the original bytes as-is.
    if len(boundaries) <= 1:
        boundary = boundaries[0] if boundaries else None
        return [
            SplitDocument(
                filename=base_filename,
                encoded_pdf=base64.b64encode(pdf_bytes).decode("utf-8"),
                boundary=boundary,
            )
        ]

    reader = PdfReader(io.BytesIO(pdf_bytes))
    page_count = len(reader.pages)

    split_documents: list[SplitDocument] = []
    for i, boundary in enumerate(boundaries, start=1):
        writer = PdfWriter()
        # start_page/end_page are 1-based inclusive; PdfReader.pages is 0-based.
        # Clamp to the real page count as a safety net against bad boundaries.
        start_idx = max(boundary.start_page - 1, 0)
        end_idx = min(boundary.end_page, page_count)
        for page_idx in range(start_idx, end_idx):
            writer.add_page(reader.pages[page_idx])

        buffer = io.BytesIO()
        writer.write(buffer)
        child_bytes = buffer.getvalue()

        split_documents.append(
            SplitDocument(
                filename=f"{stem}_{i:03d}{suffix}",
                encoded_pdf=base64.b64encode(child_bytes).decode("utf-8"),
                boundary=boundary,
            )
        )

    return split_documents


split_documents = split_pdf(pdf_bytes, attachment_path.name, segmentation.documents)

print(f"{len(split_documents)} document(s) ready for extraction:")
for doc in split_documents:
    b = doc.boundary
    print(f"  {doc.filename}  (pages {b.start_page}-{b.end_page})" if b else f"  {doc.filename}")


1 document(s) ready for extraction:
  Invoice 229555.pdf  (pages 1-1)


In [47]:
def build_extraction_human_message(filename: str, encoded_pdf: str) -> HumanMessage:
    """Build the extraction HumanMessage for a single (already segmented) document."""
    return HumanMessage(
        content=[
            {
                "type": "text",
                "text": """
Analyze the attached PDF.

Classify the document and, if it is an invoice-like document,
extract the requested structured information.

Follow the provided schema exactly.

If the document is not invoice-like, classify it as "other" and
leave extracted_data as null.
""",
            },
            {
                "type": "file",
                "file": {
                    "file_data": f"data:application/pdf;base64,{encoded_pdf}",
                    "filename": filename,
                    "format": "application/pdf",
                },
            },
        ]
    )


system_message = SystemMessage(
    content="""
You are an expert document understanding system specialized in invoices and accounting documents.

Your task is to classify the document and extract structured information into the provided schema.

GENERAL RULES

- The schema is the source of truth. Follow every field description exactly.
- Every field is optional.
- If a value cannot be determined confidently from the document, return null.
- Never invent, estimate or complete missing information.
- Prefer returning null rather than guessing.
- Use only information present in the document.
- Ignore filenames, metadata and external knowledge.

CONFIDENCE

Every extracted field must include an honest confidence score between 0 and 1.

Use the following scale:

1.00
    Explicitly visible and completely unambiguous.

0.90–0.99
    Clearly visible with only minor formatting or OCR uncertainty.

0.70–0.89
    Probably correct but there is some ambiguity.

0.50–0.69
    Weak evidence or partially inferred.

Below 0.50
    The value is too uncertain.
    Prefer returning null instead.

EVIDENCE

Every extracted value must include evidence.

Evidence must:

- be copied VERBATIM from the document;
- never be paraphrased;
- be as short as possible while uniquely supporting the value;
- preserve original spelling, punctuation and formatting whenever possible.

GOOD:
    "Petrogal, S.A."
    "Total da Fatura 1.607,70"
    "VAT Number PT511034750"

BAD:
    "The supplier is Petrogal."
    "Invoice total."

NORMALIZATION

Normalize values only when requested by the schema.

Examples:

- Dates -> DD-MM-YYYY
- Monetary values -> numeric values only
- Portuguese NIFs:
    if exactly 9 digits are shown without country prefix,
    normalize to PT#########.

Do not normalize evidence.

CLASSIFICATION

Determine the accounting nature of the document.

Examples include:

- invoice
- receipt
- credit note
- debit note
- other

Also determine the document state when explicitly visible:

- original
- proforma
- copy
- cancelled

If the document is not invoice-like, classify it as "other" and leave the invoice fields null.

SEMANTIC BLOCKS

When semantic blocks are requested:

- preserve the original reading order;
- preserve table rows;
- preserve line breaks when useful;
- exclude advertisements, decorative elements, QR codes, bank details,
  long legal disclaimers and unrelated attachments unless they are required
  to validate the extracted accounting information.

FINAL RULE

Accuracy is more important than completeness.

It is always preferable to return fewer fields with high confidence than many uncertain fields.
"""
)

In [48]:
# ---------------------------------------------------------------------------
# Node 3 — Extraction (once per split document)
#
# Build one HumanMessage per document and run the existing ClassifyExtract
# pipeline. Every extraction call receives exactly ONE document.
#
# Robustness notes:
# - This pixtral-large is served via Bedrock, which REJECTS tool_choice /
#   parallel_tool_calls. So we must NOT use method="function_calling" — it
#   fails with a 400. We stay on the default structured-output path (the same
#   one the segmentation node uses successfully).
# - Occasionally the model still replies with prose instead of JSON, which
#   surfaces as a ValidationError. We catch it per-document and retry ONCE
#   with a stricter "JSON only" nudge, so one bad document neither crashes the
#   batch nor is silently dropped.
# ---------------------------------------------------------------------------

from langchain_core.exceptions import OutputParserException
from pydantic import ValidationError

structured_llm = llm.with_structured_output(ClassifyExtract)

# Parse failures we want to retry (prose instead of JSON), as opposed to
# transport/HTTP errors which won't improve on a plain retry.
PARSE_ERRORS = (ValidationError, OutputParserException, ValueError)

json_only_nudge = HumanMessage(
    content=(
        "Your previous reply could not be parsed. Respond again with ONLY a single "
        "JSON object that matches the required schema — no prose, no Markdown, no "
        "code fences."
    )
)


def extract_document(doc: SplitDocument) -> ClassifyExtract:
    human_message = build_extraction_human_message(doc.filename, doc.encoded_pdf)
    try:
        return structured_llm.invoke([system_message, human_message])
    except PARSE_ERRORS:
        # One retry with an explicit JSON-only reminder appended.
        return structured_llm.invoke([system_message, human_message, json_only_nudge])


results: list[tuple[SplitDocument, ClassifyExtract]] = []
failures: list[tuple[SplitDocument, Exception]] = []

for doc in split_documents:
    try:
        response = extract_document(doc)
    except Exception as exc:  # noqa: BLE001 - keep the batch alive, inspect after
        failures.append((doc, exc))
        print(f"❌ Failed to extract {doc.filename}: {type(exc).__name__}: {exc}")
        continue

    results.append((doc, response))
    print(f"✅ Extracted {doc.filename}")

print(f"\n{len(results)} succeeded, {len(failures)} failed out of {len(split_documents)} document(s).")


✅ Extracted Invoice 229555.pdf

1 succeeded, 0 failed out of 1 document(s).


In [49]:
from typing import Any


def _confidence_icon(confidence: float | None) -> str:
    if confidence is None:
        return "⚪"
    if confidence >= 0.9:
        return "🟢"
    if confidence >= 0.7:
        return "🟡"
    return "🔴"


def _print_field(label: str, field: Any) -> None:
    """
    Pretty-print one Confident[T] field.

    Expected shape:
        field.value
        field.confidence
        field.evidence
    """
    if field is None:
        print(f"{label:<15} {'-':<35}")
        return

    value = str(field.value) if field.value is not None else "-"
    confidence = getattr(field, "confidence", None)

    print(
        f"{label:<15}"
        f"{value:<40}"
        f"{_confidence_icon(confidence)} "
        f"{confidence:.2f}"
    )

    evidence = getattr(field, "evidence", None)
    if evidence:
        print(f"{'':15}evidence: {evidence}")


def print_invoice_result(filename: str, result: ClassifyExtract) -> None:
    print("=" * 80)
    print(f"📄 {filename}")
    print("=" * 80)

    print("\nClassification")
    print("-" * 14)
    _print_field("Type", result.classification.document_type)
    _print_field("State", result.classification.document_state)

    invoice = result.extracted_data
    if invoice is None:
        print("\nNo invoice data extracted.")
        return
    
    print("\nParties")
    print("-" * 7)
    _print_field("Supplier", invoice.supplier_name)
    _print_field("Supplier VAT", invoice.supplier_vat)
    _print_field("Client", invoice.client_name)
    _print_field("Client VAT", invoice.client_vat)

    print("\nAmounts")
    print("-" * 7)
    _print_field("Issue date", invoice.issue_date)
    _print_field("Base", invoice.base_amount)
    _print_field("VAT", invoice.vat_amount)
    _print_field("Total", invoice.total_amount)


for doc, result in results:
    print_invoice_result(doc.filename, result)
    print()

📄 Invoice 229555.pdf

Classification
--------------
Type           invoice                                 🟢 1.00
               evidence: INVOICE
State           -                                  

Parties
-------
Supplier       GSLines Transportes Maritimos LDA       🟢 1.00
               evidence: GSLines Transportes Maritimos LDA
Supplier VAT   PT511011911                             🟢 1.00
               evidence: PT511011911
Client         5000355882                              🟢 0.90
               evidence: 5000355882
Client VAT     NL813715532B01                          🟢 0.90
               evidence: NL813715532B01

Amounts
-------
Issue date     01-08-2026                              🟢 1.00
               evidence: 01/08/2026
Base           104.0                                   🟢 1.00
               evidence: Basis VAT

104,00
VAT            0.0                                     🟢 1.00
               evidence: VAT Amount

0,00
Total          104.0                    